In [5]:
import polars as pl
import pandas as pd
import duckdb
import httpx
import pyarrow as pa
import matplotlib.pyplot as plt

print("Entorno de trabajo configurado correctamente.")

Entorno de trabajo configurado correctamente.


In [6]:
url = "https://www.datos.gov.co/api/views/jbjy-vk9h.json"

respuesta = httpx.get(url)
metadatos = respuesta.json()

print(metadatos["name"])

categoria = metadatos["columns"]
print(type(categoria))
print(len(categoria))
print(categoria[0])
print(categoria[0]["fieldName"])

for i in range(len(categoria)):
    print(f"name: {categoria[i]['name']}, fieldName: {categoria[i]['fieldName']}, dataTypeName: {categoria[i]['dataTypeName']}, description: {categoria[i]['description']}")





SECOP II - Contratos Electrónicos
<class 'list'>
95
{'id': 649573576, 'name': 'Nombre Entidad', 'dataTypeName': 'text', 'description': 'Nombre de la entidad del estado que publica el contrato', 'fieldName': 'nombre_entidad', 'position': 1, 'renderTypeName': 'text', 'tableColumnId': 88481440, 'format': {}}
nombre_entidad
name: Nombre Entidad, fieldName: nombre_entidad, dataTypeName: text, description: Nombre de la entidad del estado que publica el contrato
name: Nit Entidad, fieldName: nit_entidad, dataTypeName: number, description: NIT de la entidad del estado que publica el contrato
name: Departamento, fieldName: departamento, dataTypeName: text, description: Departamento en el cual se registró la entidad del estado que publica el contrato
name: Ciudad, fieldName: ciudad, dataTypeName: text, description: Ciudad en el cual se registró la entidad del estado que publica el contrato
name: Localización, fieldName: localizaci_n, dataTypeName: text, description: Ubicación completa de la enti

# Variables clave

## 1. Sobre la entidad y el lugar de realización
- nombre_entidad: nombre de la entidad del Estado que publica el contrato.
- nit_entidad: NIT de la entidad; identificador único para agrupar y cruzar contratos de una misma entidad.
- codigo_entidad: código de la entidad dentro de la plataforma SECOP II.
- departamento: departamento donde está registrada la entidad.
- ciudad: ciudad donde está registrada la entidad.
- orden: si la entidad es del orden nacional o territorial.
- sector: sector al que pertenece la entidad (salud, educación, defensa, etc.).
- rama: rama del poder público de la entidad (ejecutiva, legislativa, judicial).
- entidad_centralizada: si la entidad es centralizada o descentralizada.
- nombre_representante_legal: nombre del representante legal (la documentación dice "de la entidad", pero hay que verificar con datos reales si corresponde al proveedor).
- identificaci_n_representante_legal: número de identificación de ese representante legal (dato personal: no publicarlo).

## 2. Sobre el contrato
- proceso_de_compra: identificador del proceso de compra del que salió el contrato.
- id_contrato: identificador único del contrato firmado; sirve como llave para no duplicar registros.
- urlproceso: enlace al proceso en SECOP II, útil para revisar casos puntuales.
- estado_contrato: estado actual del contrato (en ejecución, modificado, suspendido, cerrado, terminado, etc.).
- liquidaci_n: si el contrato ya fue liquidado, es decir, cerrado formalmente con el balance final de cuentas.
- codigo_de_categoria_principal: código UNSPSC de la categoría del bien o servicio; sirve para agrupar por tipo de compra.
- objeto_del_contrato: qué se contrató, descrito en texto.
- descripcion_del_proceso: descripción del proceso de compra; útil para casos puntuales o análisis de texto.
- tipo_de_contrato: naturaleza jurídica del contrato (prestación de servicios, obra, suministro, compraventa, etc.).
- modalidad_de_contratacion: cómo se escogió al proveedor (licitación pública, contratación directa, mínima cuantía, selección abreviada, concurso de méritos, etc.).
- justificacion_modalidad_de: razón dada para usar esa modalidad; clave para el análisis crítico de la contratación directa.
- habilita_pago_adelantado: si el contrato permite entregar dinero por adelantado (anticipo) al proveedor.
- duraci_n_del_contrato: duración pactada en texto (por ejemplo "87 Dia(s)" o "11 Mes(es)"); hay que convertirla a número de días.
- dias_adicionados: días que se le sumaron al contrato después de firmado (prórrogas).

## 3. Fechas
- fecha_de_firma: día en que se firmó el contrato.
- fecha_de_inicio_del_contrato: día en que empiezan las obligaciones del contrato; no debería ser anterior a la firma.
- fecha_de_fin_del_contrato: día en que terminan las obligaciones del contrato.

## 4. Proveedores
- proveedor_adjudicado: nombre de la persona o empresa que ganó el contrato.
- tipodocproveedor: tipo de documento del proveedor (cédula de ciudadanía, NIT, etc.).
- documento_proveedor: número de documento del proveedor; identificador para agrupar todos sus contratos.
- codigo_proveedor: código del proveedor dentro de la plataforma SECOP II.
- es_pyme: si el proveedor es una pequeña o mediana empresa.
- es_grupo: si el proveedor es un consorcio o unión temporal (un grupo de empresas).

## 5. Información presupuestal
- origen_de_los_recursos: de dónde salen los recursos a nivel presupuestal.
- destino_gasto: en qué se gastan los recursos (funcionamiento o inversión).
- presupuesto_general_de_la_nacion_pgn: parte del valor que viene del Presupuesto General de la Nación.
- sistema_general_de_participaciones: parte del valor que viene del Sistema General de Participaciones (transferencias de la Nación a departamentos y municipios).
- sistema_general_de_regal_as: parte del valor que viene del Sistema General de Regalías.
- valor_del_contrato: valor total del contrato.
- valor_de_pago_adelantado: dinero entregado por adelantado al proveedor.
- valor_amortizado: parte del anticipo que el proveedor ya justificó con trabajo o entregas.
- valor_pendiente_de: parte del anticipo que falta por justificar (pendiente de amortización); si es alto y el contrato está parado, hay dinero en riesgo.
- valor_facturado: lo que el proveedor ha cobrado hasta la fecha.
- valor_pagado: lo que la entidad ha pagado hasta la fecha.
- valor_pendiente_de_pago: lo que la entidad aún debe pagar.
- valor_pendiente_de_ejecucion: lo que falta por ejecutar del contrato; si es alto en un contrato vencido, puede indicar una obra o servicio inconcluso.


# Borrador de indicadores (generado con apoyo de IA, por revisar)

Módulo 1: Concentración y destino del gasto (La radiografía general)

Objetivo: Entender quién gasta, en qué sectores se concentra el dinero y bajo qué excusas se evita la competencia abierta.
• Variables: nombre_entidad, nit_entidad, departamento, orden, sector, entidad_centralizada, codigo_de_categoria_principal, descripcion_del_proceso, tipo_de_contrato, modalidad_de_contratacion, justificacion_modalidad_de, origen_de_los_recursos, destino_gasto, espostconflicto, valor_del_contrato.
• Métricas clave a calcular:
	• Porcentaje de contratación a dedo: Proporción del dinero total adjudicado por Contratación directa o Régimen especial frente a Licitación pública.
	• Concentración por sector (Pareto 80/20): Qué códigos UNSPSC (codigo_de_categoria_principal) y qué entidades concentran el 80 % del presupuesto.
	• Foco en entidades descentralizadas: Comparar si las entidades marcadas como Descentralizada usan más contratación directa que las Centralizada.

Módulo 2: Riesgo de pérdida de recursos y sobrecostos (Auditoría financiera)

Objetivo: Detectar dinero público entregado sin respaldo, anticipos en riesgo y obras o proyectos abandonados.
• Variables: id_contrato, estado_contrato, liquidaci_n, valor_del_contrato, habilita_pago_adelantado, valor_de_pago_adelantado, valor_facturado, valor_pagado, valor_amortizado, valor_pendiente_de (pendiente de amortizar), valor_pendiente_de_ejecucion, fecha_de_fin_del_contrato, urlproceso.
• Banderas rojas (Red Flags) a programar:
	• Anticipos en riesgo: Contratos con habilita_pago_adelantado == "Sí" y valor_pendiente_de > 0 cuyo estado_contrato sea Suspendido, Terminado anormalmente o que ya hayan superado su fecha_de_fin_del_contrato.
	• Contratos fantasma o inconclusos: Contratos cuya fecha de fin ya pasó hace más de 6 meses, siguen figurando En ejecución (o Terminado sin liquidaci_n) y tienen un valor_pendiente_de_ejecucion alto.
	• Pagos sin facturación: Registros donde valor_pagado > valor_facturado (inconsistencia contable o pagos irregulares).

Módulo 3: Favoritismo y anomalías de planeación (Manipulación de reglas)

Objetivo: Identificar contratos diseñados a la medida, fraccionamiento para evadir licitaciones y urgencias sospechosas.
• Variables: nit_entidad, documento_proveedor, proveedor_adjudicado, modalidad_de_contratacion, justificacion_modalidad_de, codigo_de_categoria_principal, fecha_de_firma, fecha_de_inicio_del_contrato, fecha_de_fin_del_contrato, dias_adicionados, valor_del_contrato.
• Banderas rojas (Red Flags) a programar:
	• Índice de concentración de proveedores (HHI): Medir si una alcaldía o entidad le entrega más del 30 % o 40 % de su presupuesto a un solo documento_proveedor.
	• Fraccionamiento de contratos: Detectar pares (nit_entidad, documento_proveedor) que firman 3 o más contratos de Mínima cuantía o Contratación directa bajo el mismo codigo_de_categoria_principal en un rango menor a 30 días.
	• Prórrogas desproporcionadas: Calcular el ratio dias_adicionados / duracion_inicial_dias. Si un contrato dura el doble o triple de lo planeado inicialmente, indica mala planeación o ventaja injusta frente a otros competidores.
	• Contratos exprés y hechos cumplidos: Casos donde fecha_de_inicio_del_contrato < fecha_de_firma (empezaron antes de firmar) o picos de firma en los últimos 15 días de diciembre o justo antes de Ley de Garantías.

Módulo 4: Redes de corrupción y colusión (Análisis de grafos)

Objetivo: Descubrir carteles de contratación, empresas fachada y grupos que acaparan múltiples entidades.
• Variables: nit_entidad, nombre_entidad, documento_proveedor, proveedor_adjudicado, tipodocproveedor, es_grupo, es_pyme, nombre_representante_legal, identificaci_n_representante_legal, valor_del_contrato.
• Análisis de redes (NetworkX en Python) a construir:
	• Carrusel de empresas fachada: Buscar un mismo identificaci_n_representante_legal asociado a dos o más documento_proveedor (NIT) diferentes que contratan con la misma entidad (simulación de competencia).
	• Proveedores multi-entidad (Nodos centrales): Graficar una red bipartita (Entidad ↔ Proveedor) ponderada por valor_del_contrato para encontrar contratistas que ganan simultáneamente en decenas de municipios o ministerios.
	• Consorcios y Uniones Temporales (es_grupo == "Sí"): Filtrar las megaobras adjudicadas a grupos empresariales para auditar si los mismos consorcios rotan en las mismas regiones.